# CorrDiff - Fase 8 v2 - Persistência + Auditoria de Continuidade

Baselines temporais e auditoria direta de target(t) versus target(t-1h).

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('../analysis_outputs/08_persistence_baselines')
summary = json.loads((OUT/'analysis_summary.json').read_text())
summary


## 1. Cohort e split cronológico

In [ ]:
cohort = pd.read_parquet(OUT/'evaluation_cohort_summary.parquet')
display(cohort)


## 2. Brier Skill Score no teste

In [ ]:
binary = pd.read_parquet(OUT/'binary_baseline_metrics.parquet')
test = binary[binary.split.eq('test')]
for event_id in ['ge_30','ge_40','ge_45']:
    t = test[test.event_id.eq(event_id)].sort_values('brier_skill_vs_climatology', ascending=False)
    display(t[['event_id','baseline','brier','brier_skill_vs_climatology','roc_auc','average_precision','ece_10bins']])
    fig, ax = plt.subplots(figsize=(9,4))
    ax.bar(t.baseline, t.brier_skill_vs_climatology)
    ax.axhline(0)
    ax.set_ylabel('Brier Skill Score vs climatologia')
    ax.set_title(event_id)
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()


## 3. Skill contínuo no teste

In [ ]:
cont = pd.read_parquet(OUT/'continuous_baseline_metrics.parquet')
ct = cont[cont.split.eq('test')]
for metric in ['max_dbz','positive_pixel_fraction','event_pixel_fraction_ge_40','event_pixel_fraction_ge_45']:
    t = ct[ct.metric.eq(metric)].sort_values('rmse_skill_vs_climatology', ascending=False)
    display(t[['metric','baseline','mae','rmse','mae_skill_vs_climatology','rmse_skill_vs_climatology','pearson_r']])


## 4. Métricas condicionadas a eventos intensos

In [ ]:
cond = pd.read_parquet(OUT/'continuous_baseline_metrics_by_condition.parquet')
display(cond[(cond.split.eq('test')) & (cond.condition.isin(['ge_30','ge_40','ge_45']))])


## 5. Skill por estação

In [ ]:
bs = pd.read_parquet(OUT/'binary_baseline_metrics_by_season.parquet')
display(bs[(bs.split.eq('test')) & (bs.event_id.eq('ge_45'))][['season_code','baseline','n','event_rate','brier','roc_auc','average_precision']])


## Interpretação

O baseline principal de referência é a climatologia mês × hora local ajustada apenas no treino. Skill positivo indica ganho sobre essa referência. Para >=45 dBZ, priorize Brier, Average Precision e calibração, e não apenas ROC AUC.

## 6. Auditoria de igualdade exata do target

In [ ]:
audit = pd.read_parquet(OUT/'radar_continuity_summary.parquet')
display(audit)


## 7. Auditoria por ano

In [ ]:
year = pd.read_parquet(OUT/'radar_continuity_by_year.parquet')
display(year[year.condition.isin(['any_wet','either_ge_30','either_ge_40','either_ge_45'])])
for condition in ['any_wet','either_ge_40','either_ge_45']:
    t = year[year.condition.eq(condition)].copy()
    fig, ax = plt.subplots(figsize=(10,4))
    ax.plot(t.year_utc.astype(int), t.exact_target_equal_ratio, marker='o')
    ax.set_xlabel('Ano UTC')
    ax.set_ylabel('Razão de targets exatamente iguais em 1 h')
    ax.set_title(condition)
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.show()


## 8. Runs de targets idênticos

In [ ]:
runs = pd.read_parquet(OUT/'radar_identical_target_runs.parquet')
display(runs[~runs.dry_target].sort_values('length_timestamps', ascending=False).head(30))
